# Supplementary Figure 21: asymmetry of the causal graphs under the modified CaLLTiF threshold

For every NF and baseline graph, the graph asymmetry $\|A - A^T\|_1 / \|A + A^T\|_1$ is computed while the
significance threshold for lagged edges (between nodes that already have a contemporaneous edge) is relaxed
from the original $\alpha = 0.0025$. The paper uses the relaxed value $\bar\alpha = 0.05$.

**Inputs:** the CaLLTiF graphs from `2_causal_graphs/` (`Results/calltif_decnef_study{N}_{nf,dec}_two_runs_one_graph.npz`).
**Output:** `Results/figures_supplementary/SuppFig21_modified_calltif_asymmetry.{pdf,png}`.


In [ ]:
# Run from the repository root so that 'Results/...' paths and the shared modules resolve
import os, sys
ROOT = os.path.dirname(os.path.abspath(''))
os.chdir(ROOT)
sys.path.insert(0, ROOT)

# Output folder for the figure
FIG_SUPP = 'Results/figures_supplementary'
os.makedirs(FIG_SUPP, exist_ok=True)

# Imports

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import csv
import pandas as pd
from scipy.stats import pearsonr
from scipy.spatial import distance_matrix
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from scipy import stats
import seaborn as sb
import glob
from scipy.io import savemat
import importlib
import itertools
import scipy.io
import networkx as nx
import os
import datetime
import time
import sys
import warnings
import seaborn as sns
import functions as funcs
from sklearn.metrics import pairwise_distances
import matplotlib.colors as mcolors

importlib.reload(funcs)

from matplotlib.ticker import PercentFormatter
from nilearn.connectome import ConnectivityMeasure
from nilearn import plotting
import matplotlib.cm as cm
import matplotlib.colors as colors
import nilearn
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import itertools
from matplotlib.colors import BoundaryNorm

import pingouin as pg

from statsmodels.stats.multitest import multipletests
import statsmodels

# Variables

In [ ]:
num_parcels = 116
num_cortical_parcels = 100
num_subcortical_parcels = 16

num_nf_sessions = 3
num_decoder_sessions = 1
subject_id_selected = np.array(['01','02','03','04','05','06','07','08','09']) 
num_subjects = subject_id_selected.shape[0]

parcellation_mask = np.asanyarray(nib.load('External Packages and Files/Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1.dlabel.nii').dataobj)

# network
num_networks = 16

network_dic = "{'LH_Vis':[],'RH_Vis':[], 'LH_Sub' :[],'RH_Sub' :[],'LH_Def':[],'RH_Def':[],'LH_Dor':[],'RH_Dor':[],'LH_Sal':[],'RH_Sal':[],'LH_Som':[],'RH_Som':[],'LH_Lim':[],'RH_Lim':[], 'LH_Con':[],'RH_Con':[]}"
net_labels = ['LH_Vis','RH_Vis', 'LH_Sub','RH_Sub','LH_Def','RH_Def','LH_Dor','RH_Dor','LH_Sal','RH_Sal','LH_Som','RH_Som','LH_Lim','RH_Lim', 'LH_Con','RH_Con']

visualization_labels = ['L-Vis','R-Vis','L-Sub','R-Sub','L-DMN','R-DMN','L-Dors-Att','R-Dors-Att','L-Vent-Att','R-Vent-Att','L-Som-Mot','R-Som-Mot', 'L-Lim','R-Lim','L-Cont','R-Cont']

num_networks = len(net_labels)

with open('Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1_label.txt') as f:
    lines = f.readlines()

parcel_labels = []
parcel_locs = []
i = 0
while i< 2*num_parcels:
    if i> 31:
        parcel_labels.append(lines[i].strip('\n')[10:])
    else:
        parcel_labels.append(lines[i].strip('\n'))
    i = i+2

print(parcel_labels)

# CaLLTiF
alpha_level = 1e-2/4 # tau_max = 1 ----> alpha/(tau_max+1)*2^(tau_max)
tau_max = 1

num_subjects_ready = 9

num_samples_in_each_run = 154      #175
num_trials_each_run = 15
num_dec_runs  = 5      ### five consecutive pairs of runs (6 runs)
num_nf_runs = 7        ### seven consecutive pairs of runs (8 runs)

# Load CaLLTiF graphs

In [ ]:
def import_npz(npz_file):
    Data = np.load(npz_file, allow_pickle= True)
    for varName in Data:
        globals()[varName] = Data[varName]  

In [ ]:
import_npz('Results/calltif_decnef_study1_dec_two_runs_one_graph.npz')
import_npz('Results/calltif_decnef_study1_nf_two_runs_one_graph.npz')

import_npz('Results/calltif_decnef_study2_dec_two_runs_one_graph.npz')
import_npz('Results/calltif_decnef_study2_nf_two_runs_one_graph.npz')

import_npz('Results/calltif_decnef_study3_dec_two_runs_one_graph.npz')
import_npz('Results/calltif_decnef_study3_nf_two_runs_one_graph.npz')

import_npz('Results/calltif_decnef_study4_dec_two_runs_one_graph.npz')
import_npz('Results/calltif_decnef_study4_nf_two_runs_one_graph.npz')

import_npz('Results/calltif_decnef_study5_dec_two_runs_one_graph.npz')
import_npz('Results/calltif_decnef_study5_nf_two_runs_one_graph.npz')


# Graph asymmetry across the relaxed lagged threshold

In [ ]:
def compute_asymmetry_index(matrix):
    # Ensure the matrix is binary
    assert np.all(np.logical_or(matrix == 0, matrix == 1)), "Matrix must be binary"

    # Compute the transpose of the matrix
    transpose_matrix = matrix.T

    # Compute the numerator |A - A^T|
    numerator = np.sum(np.abs(matrix - transpose_matrix))

    # Compute the denominator |A + A^T|
    denominator = np.sum(matrix + transpose_matrix)

    # Calculate the asymmetry index
    asymmetry_index = numerator / denominator

    return asymmetry_index

In [ ]:
control_parameter = list(range(1, 401, 10))

In [ ]:
# Asymmetry of every NF and baseline graph as a function of the relaxed lagged threshold.
# control_parameter scales the threshold for lagged edges between nodes that already have a
# contemporaneous edge (relaxed alpha = control_parameter * alpha_level).
# Computed once and cached in Results/, since it takes a while.
ASYM_CACHE = 'Results/causal_graph_asymmetries_data_new.npz'

def graph_asymmetries(link_matrices, n_sessions, n_graphs):
    """One row per graph (order: subject, session, run pair, study), one column per control_parameter."""
    rows = []
    for sub in range(num_subjects_ready):
        for ses in range(n_sessions):
            for r in range(n_graphs):
                for link_matrix in link_matrices:
                    g = link_matrix[sub][ses][r]
                    p_matrix, statistics = np.array(g['p_matrix']), np.array(g['val_matrix'])
                    graph = funcs.convert_to_string_graph(p_matrix <= alpha_level)
                    row = []
                    for c in control_parameter:
                        binary_matrix, _, _ = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_test(
                            graph, p_matrix, statistics, alpha_level, c)
                        row.append(compute_asymmetry_index(binary_matrix))
                    rows.append(row)
    return np.array(rows)

if os.path.exists(ASYM_CACHE):
    import_npz(ASYM_CACHE)
else:
    all_binary_decoder_causal_graphs_assymetry = graph_asymmetries(
        [multipledataset_calltif_decoder_study1_link_matrix, multipledataset_calltif_decoder_study2_link_matrix,
         multipledataset_calltif_decoder_study3_link_matrix, multipledataset_calltif_decoder_study4_link_matrix,
         multipledataset_calltif_decoder_study5_link_matrix], num_decoder_sessions, num_dec_runs)
    all_binary_nf_causal_graphs_assymetry = graph_asymmetries(
        [multipledataset_calltif_nf_study1_link_matrix, multipledataset_calltif_nf_study2_link_matrix,
         multipledataset_calltif_nf_study3_link_matrix, multipledataset_calltif_nf_study4_link_matrix,
         multipledataset_calltif_nf_study5_link_matrix], num_nf_sessions, num_nf_runs)
    np.savez(ASYM_CACHE, all_binary_nf_causal_graphs_assymetry=all_binary_nf_causal_graphs_assymetry,
             all_binary_decoder_causal_graphs_assymetry=all_binary_decoder_causal_graphs_assymetry)

# Figure

In [ ]:
# combine all the graphs

all_binary_causal_graphs_assymetry_both_nf_dc = np.concatenate((all_binary_nf_causal_graphs_assymetry, all_binary_decoder_causal_graphs_assymetry), axis = 0)
# Plot
plt.figure(figsize=(12, 6))

plt.plot(np.array(control_parameter)*alpha_level, all_binary_causal_graphs_assymetry_both_nf_dc.T, color = 'lightgray')
plt.plot(np.array(control_parameter)*alpha_level, np.mean(all_binary_causal_graphs_assymetry_both_nf_dc, axis = 0), color = 'black', linewidth = 3)

plt.grid(True)
plt.xticks(rotation=30, fontsize = 16)
plt.yticks(fontsize = 16)

plt.xlabel(r'Modified $ \alpha$', fontsize= 24)
plt.ylabel('Causal Graph Asymmetry', fontsize = 24)

plt.axvline(x=alpha_level, color='blue', linestyle='--', linewidth = 3)
plt.axvline(x=alpha_level*20, color='red', linestyle='--',linewidth = 3)

plt.text(alpha_level+0.0018, 0.35, 'Original \n' + r'$(\alpha = 0.0025)$', fontsize=20, ha='center', va='center', color= 'blue')
plt.text(alpha_level*20+0.08, 0.35, 'Modified value used \n'+r'$(\alpha = 0.05)$', fontsize=20, ha='center', va='center', color = 'red')

plt.xscale('log')

plt.tight_layout()

plt.savefig(os.path.join(FIG_SUPP, 'SuppFig21_modified_calltif_asymmetry.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig21_modified_calltif_asymmetry.png'), bbox_inches='tight')
